# 问题2：单Agent风险知识图谱可复核演示

本 Notebook 只实例化比赛系统原有的 `OptionRiskAgent`。用户入口仍是“品种 → 该品种有效时间点 → 确认”；知识图谱是同一结果中的可解释层，不创建第二个业务Agent，也不提供自由文本问答。演示事件是白银 `ag` 在2025-04-07 10:15:00（赛题官方极端事件区间内）的真实期权快照。

In [1]:
from pathlib import Path
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import display
from option_risk import AgentConfig, OptionRiskAgent
from option_risk.graph_queries import GraphQueryService
from option_risk.knowledge_graph import graph_from_node_link
from option_risk.knowledge_schema import FEATURE_RULES, KG_SCHEMA_VERSION, KG_RULE_VERSION

ROOT = Path.cwd()
if not (ROOT / 'src' / 'option_risk').exists():
    ROOT = ROOT.parent
data_value = os.environ.get('OPTION_RISK_DATA_DIR')
archive_value = os.environ.get('OPTION_RISK_ARCHIVE_PATH')
fallback = ROOT / 'data' / 'demo'
if data_value:
    config = AgentConfig(data_source=Path(data_value), cache_dir=ROOT / 'data' / 'notebook_cache', output_dir=ROOT / 'outputs' / 'demo')
elif archive_value:
    config = AgentConfig(archive_path=Path(archive_value), cache_dir=ROOT / 'data' / 'notebook_cache', output_dir=ROOT / 'outputs' / 'demo')
elif fallback.exists():
    config = AgentConfig(data_source=fallback, cache_dir=ROOT / 'data' / 'notebook_cache', output_dir=ROOT / 'outputs' / 'demo')
else:
    raise FileNotFoundError('Set OPTION_RISK_DATA_DIR or OPTION_RISK_ARCHIVE_PATH')
agent = OptionRiskAgent(config)
print('PUBLIC_AGENT=OptionRiskAgent')


PUBLIC_AGENT=OptionRiskAgent


In [2]:
products = [item.code for item in agent.list_products()]
selected_product = 'ag'
assert selected_product in products
available_times = agent.list_timestamps(selected_product)
selected_time = pd.Timestamp('2025-04-07 10:15:00')
assert selected_time in available_times
pd.Series({'actual_product_count': len(products), 'selected_product': selected_product, 'valid_timestamp_count': len(available_times), 'selected_timestamp': selected_time.isoformat()}, name='只读下拉选择')


actual_product_count                       1
selected_product                          ag
valid_timestamp_count                   1959
selected_timestamp       2025-04-07T10:15:00
Name: 只读下拉选择, dtype: object

In [3]:
result = agent.analyze(selected_product, selected_time)
graph_result = result.knowledge_graph
assert result.status == 'ready'
assert graph_result is not None and graph_result.status == 'ready'
assert result.product == graph_result.product == selected_product
assert result.timestamp == graph_result.timestamp == selected_time
print(f'REAL_EVENT={selected_product}/{selected_time.isoformat()}')


REAL_EVENT=ag/2025-04-07T10:15:00


/Users/xuji/Documents/第五届中国研究生金融科技创新大赛/学习拆解工作区/隔离复现项目/option_risk_warning/src/option_risk/agent.py:602: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  candidate = pd.concat([prefix, features], axis=0)
/Users/xuji/Documents/第五届中国研究生金融科技创新大赛/学习拆解工作区/隔离复现项目/option_risk_warning/src/option_risk/decision.py:234: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ready = data_ready.fillna(False) & history_ready.fillna(False) & core.notna()


In [4]:
problem1 = pd.Series({
    'alert_level': result.alert_level, 'alert_name': result.alert_name,
    'alert_emitted': result.alert_emitted, 'risk_score': result.risk_score,
    'policy_version': result.policy_version, 'policy_parameter_hash': result.policy_parameter_hash,
    'transition_reason': result.transition_reason, 'price_source': result.price_source,
}, name='问题1正式结果')
display(problem1)
metrics = ['iv', 'delta', 'gamma', 'vega', 'theta']
fig = plt.figure(figsize=(14, 7), constrained_layout=True)
for position, metric in enumerate(metrics, start=1):
    surface = result.surfaces[metric]
    axis = fig.add_subplot(2, 3, position, projection='3d')
    x, y = __import__('numpy').meshgrid(surface.log_moneyness, surface.time_to_expiry)
    axis.plot_surface(x, y, surface.values, cmap='Blues', linewidth=0, antialiased=True)
    axis.set_title(metric.upper())
    axis.set_xlabel('ln(K/F)'); axis.set_ylabel('T'); axis.set_zlabel(metric)
display(fig)
plt.close(fig)


alert_level                                                              2
alert_name                                                              警告
alert_emitted                                                        False
risk_score                                                        4.068085
policy_version                                  validated-dynamic-state-v2
policy_parameter_hash    df74c842c88bc54a54d0f57a40aa704cfcb7177bcd3929...
transition_reason                                             warning_held
price_source                                           reconstructed_proxy
Name: 问题1正式结果, dtype: object

<Figure size 1400x700 with 5 Axes>

In [5]:
assert graph_result.schema_version == KG_SCHEMA_VERSION
assert graph_result.rule_version == KG_RULE_VERSION
mapping = pd.DataFrame([{
    'order': rule.order, 'feature': rule.feature_name, 'Evidence': rule.label,
    'Anomaly': rule.anomaly, 'Mechanism': rule.mechanism,
    'Conclusion': rule.conclusion, 'boundary': rule.boundary,
} for rule in FEATURE_RULES])
print(f'SCHEMA={graph_result.schema_version}; RULE={graph_result.rule_version}; MAPPINGS={len(mapping)}')
display(mapping)


SCHEMA=kg-schema-v1; RULE=kg-rule-v1; MAPPINGS=12


,order,feature,Evidence,Anomaly,Mechanism,Conclusion,boundary
0,1,atm_iv,ATM隐含波动率,ATM IV异常抬升,波动率整体重定价,短期波动风险上升,不预测标的涨跌
1,2,atm_iv_chg_4,ATM隐含波动率一小时变化,ATM IV短时变化异常,波动率重定价加速,短时波动敏感性上升,不断言外部事件原因
2,3,term_inversion,期限结构倒挂程度,期限结构倒挂,近端风险压力集中,短期波动风险上升,不等同于某条新闻导致
3,4,convexity_violation_ratio,曲面凸性违反率,执行价凸性违反异常,曲面定价一致性下降,估值一致性风险,不声称存在可执行套利
4,5,max_local_iv_jump,局部IV跳变,局部IV跳变,曲面局部稳定性下降,曲面估值稳定性风险,需结合覆盖和流动性理解
5,6,put_call_parity_p95_error,Put-Call平价95%偏离,Put-Call平价偏离异常,相对定价一致性下降,估值一致性风险,不声称存在可执行套利
6,7,calendar_variance_violation_ratio,日历总方差违反率,日历总方差约束违反,期限方向定价一致性下降,估值一致性风险,不等同于确定套利机会
7,8,gamma_hhi,Gamma持仓加权集中度,Gamma集中异常,非线性敏感性集中,Gamma敏感性风险增强,使用“可能放大”，不提仓位建议
8,9,vega_hhi,Vega持仓加权集中度,Vega集中异常,波动率敏感性集中,Vega敏感性风险增强,使用“可能放大”，不提仓位建议
9,10,zero_volume_share,零成交量合约占比,零成交合约占比异常,市场活跃度/可观测性受限,解释置信度降低,不支持市场风险等级上调


In [6]:
selected_paths = list(graph_result.active_paths[:3])
graph = graph_from_node_link(graph_result.graph_json)
selected_nodes = []
for path in selected_paths:
    for node_id in (path.evidence_node_id, path.anomaly_node_id, path.mechanism_node_id, path.conclusion_node_id):
        if node_id not in selected_nodes:
            selected_nodes.append(node_id)
selected_edges = [(u, v) for path in selected_paths for u, v in zip((path.evidence_node_id, path.anomaly_node_id, path.mechanism_node_id), (path.anomaly_node_id, path.mechanism_node_id, path.conclusion_node_id))]
positions = {}
for track, path in enumerate(selected_paths):
    for layer, node_id in enumerate((path.evidence_node_id, path.anomaly_node_id, path.mechanism_node_id, path.conclusion_node_id)):
        positions.setdefault(node_id, (layer, -track))
fig_graph, axis_graph = plt.subplots(figsize=(13, 4), constrained_layout=True)
if selected_nodes:
    nx.draw_networkx_nodes(graph, positions, nodelist=selected_nodes, node_color='#dbeafe', edgecolors='#235a97', node_size=2400, ax=axis_graph)
    nx.draw_networkx_edges(nx.DiGraph(selected_edges), positions, edge_color='#596776', arrows=True, arrowsize=16, ax=axis_graph)
    labels = {node_id: f"{graph.nodes[node_id]['node_type']}\n{graph.nodes[node_id].get('feature_name', graph.nodes[node_id].get('rule_id', 'result'))}" for node_id in selected_nodes}
    nx.draw_networkx_labels(graph, positions, labels=labels, font_size=7, ax=axis_graph)
else:
    axis_graph.text(0.5, 0.5, 'No active anomaly path', ha='center', va='center')
axis_graph.set_title('Evidence -> Anomaly -> Mechanism -> Conclusion (top 3 paths)')
axis_graph.axis('off')
display(fig_graph)
plt.close(fig_graph)
assert len(selected_paths) <= 3
print('FOUR_LAYERS=Evidence -> Anomaly -> Mechanism -> Conclusion')


<Figure size 1300x400 with 1 Axes>

FOUR_LAYERS=Evidence -> Anomaly -> Mechanism -> Conclusion


In [7]:
queries = GraphQueryService(graph_result)
root_rows = queries.root_evidence(limit=3)
alert_paths = queries.paths_to_alert()
consequences = queries.possible_consequences()
quality_paths = queries.quality_limitations()
node_detail = queries.inspect_node(root_rows[0]['node_id']) if root_rows else queries.inspect_node(graph_result.graph_json['nodes'][0]['node_id'])
anomaly_paths = queries.paths_from_anomaly(alert_paths[0]['anomaly_node_id']) if alert_paths else []
pd.Series({'root_evidence': len(root_rows), 'paths_to_alert': len(alert_paths),
'possible_consequences': len(consequences), 'quality_limitations': len(quality_paths),
'inspect_node': node_detail['node_id'], 'paths_from_anomaly': len(anomaly_paths)}, name='固定预设查询')


root_evidence                                                            3
paths_to_alert                                                           6
possible_consequences                                                    6
quality_limitations                                                      0
inspect_node             evidence:ag:20250407T101500:put_call_parity_p9...
paths_from_anomaly                                                       2
Name: 固定预设查询, dtype: object

In [8]:
assert len(graph_result.explanation_sections) == 5
for section in graph_result.explanation_sections:
    print(f'## {section.title}\n{section.content}\n')


## 当前结论
产品 AG，所选时间 2025-04-07 10:15:00；WarningState 为0—3：警告（等级2），风险分数 4.06809，状态转换原因 warning_held。WarningState 由问题一直接复制，图谱不改变该状态。

## 直接数据根因
[path:ag:20250407T101500:put_call_parity_p95_error:riskconsequence:risk_consequence_put_call_parity_p95_error] Put-Call平价95%偏离：当前值 0.077674，动态历史参考/阈值 0.0190935，压力比 4.06809，方向 下降，超过自身动态阈值。
[path:ag:20250407T101500:calendar_variance_violation_ratio:riskconsequence:risk_consequence_calendar_variance_violation_ratio] 日历总方差违反率：当前值 0.236025，动态历史参考/阈值 0.0756949，压力比 3.11811，方向 下降，当前数据出现相应约束违反。
[path:ag:20250407T101500:term_inversion:riskconsequence:risk_consequence_term_inversion] 期限结构倒挂程度：当前值 0.108707，动态历史参考/阈值 0.0454114，压力比 2.39383，方向 上升，超过自身动态阈值。

## 风险传导过程
[path:ag:20250407T101500:put_call_parity_p95_error:riskconsequence:risk_consequence_put_call_parity_p95_error] Put-Call平价95%偏离超过自身动态阈值Put-Call平价偏离异常；Put-Call平价偏离异常为该判断提供支持相对定价一致性下降；相对定价一致性下降可能带来估值一致性风险。
[path:ag:20250407T101500:put_call_parity_p95_error:warningstate:warning_2] Put-Call平价95%偏离超过自身动态阈

In [9]:
active_path_ids = {path.path_id for path in graph_result.active_paths}
referenced_path_ids = {path_id for section in graph_result.explanation_sections for path_id in section.path_ids}
assert referenced_path_ids <= active_path_ids
assert all(path_id in graph_result.query_index['paths_from_anomaly'].get(path.anomaly_node_id, [])
           for path in graph_result.active_paths for path_id in [path.path_id])
print(f'TRACEABILITY_OK={len(referenced_path_ids)}/{len(active_path_ids)}')


TRACEABILITY_OK=12/12


## 完成

该演示从真实数据选择项出发，保留问题1结论，随后构建和查询NetworkX四层图谱。所有解释路径均已回指活动路径；质量证据只降低解释置信度，不提高市场风险等级。系统不预测标的涨跌方向，也不提供交易建议。